# DART 공시 → 임베딩용 JSONL 수집 파이프라인 (노트북)

`pipeline.py` 와 동일한 흐름을 셀 단위로 분리. 디버깅·중간 점검용.

## 흐름
1. 설정 확인 (config.py / companies.py)
2. corp_code 매핑
3. 회사 리스트 확정
4. DART 다운로드 → 정제 → 청킹 → JSONL 저장 (회사별 루프)
5. 결과 요약

## 변경 포인트
- **회사 추가/삭제** → `companies.py` 의 `COMPANIES` 리스트
- **보고서 종류** (사업보고서·반기·분기) → `config.py` 의 `REPORT_TYPES`
- **회계연도** → `config.py` 의 `TARGET_FISCAL_YEAR`

---
## 1. 설정 확인

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # data_collection 폴더에서 실행 가정

import config as cfg
from companies import COMPANIES, validate

print(f"TARGET_FISCAL_YEAR : {cfg.TARGET_FISCAL_YEAR}")
print(f"REPORT_TYPES       : {cfg.REPORT_TYPES}")
print(f"검색 기간          : {cfg.get_search_window()}")
print(f"회사 수            : {len(COMPANIES)}")
print(f"RAG (JSONL)        : {cfg.RAG_DIR}")
print(f"raw (XML)          : {cfg.RAW_DIR}")

# 무결성 검증
res = validate()
if res["ok"]:
    print(f"\n✅ companies.py 무결성 OK ({res['n_total']}개사, {res['n_groups']}그룹)")
else:
    print(f"\n❌ 오류:")
    for e in res["errors"]:
        print(f"  - {e}")

# 보고서별 매칭 라벨·파일명 확인
print(f"\n각 보고서 매칭 라벨·파일명:")
for rt in cfg.REPORT_TYPES:
    label = cfg.expected_report_label(rt)
    fn = cfg.jsonl_filename("064350", "현대로템", rt)
    print(f"  {rt:<8} → '{label}' 매칭  →  {fn}")


---
## 2. corp_code 매핑

DART corpCode.xml 다운로드 (1회만, 캐시됨).

In [ ]:
from dart_downloader import DART_API_KEY, get_corp_map

assert DART_API_KEY, "DART_API_KEY 미설정 — .env 또는 환경변수 등록 후 재시작"
print(f"API key 로드 OK (len={len(DART_API_KEY)})")

stock_to_corp = get_corp_map()
print(f"corp_code 매핑: {len(stock_to_corp):,}건")
# 샘플
for sc in ["005930", "064350", "047810"]:
    if sc in stock_to_corp:
        print(f"  {sc} → {stock_to_corp[sc]}")


---
## 3. 회사 리스트 확정 (corp_code 매핑 가능 여부 확인)

In [ ]:
mapped, unmapped = [], []
for grp, sc, name in COMPANIES:
    info = stock_to_corp.get(sc)
    if info:
        mapped.append({"group": grp, "stock_code": sc, "name": name,
                       "corp_code": info["corp_code"]})
    else:
        unmapped.append((grp, sc, name))

print(f"매핑 성공: {len(mapped)} / {len(COMPANIES)}")
if unmapped:
    print("매핑 실패:")
    for u in unmapped:
        print(f"  - {u}")
mapped[:3]


---
## 4. DART 다운로드 + 청킹 + JSONL 저장 (메인 루프)

⚠ 첫 실행 시 약 5초/회사 (다운로드 + 청킹). 50개사 → 4~5분.
재실행은 이미 있는 JSONL 자동 스킵 → 즉시.

In [ ]:
import time, json, re
from datetime import datetime
from dart_downloader import fetch_reports, download_document
from xml_cleaner import clean_xml, parse_with_fallback
from chunker import build_chunks

def safe_filename(s):
    return re.sub(r'[\\/:*?"<>|&\s]+', '_', s).strip('_')

def process_one_report(co, rt, report_meta, skip_existing=True):
    sc, name = co["stock_code"], co["name"]
    safe = safe_filename(name)
    cfg.RAG_DIR.mkdir(parents=True, exist_ok=True)
    cfg.RAW_DIR.mkdir(parents=True, exist_ok=True)

    out_jsonl = cfg.RAG_DIR / cfg.jsonl_filename(sc, safe, rt)
    co_dir    = cfg.RAW_DIR / f"{sc}_{safe}"
    co_dir.mkdir(exist_ok=True, parents=True)

    if skip_existing and out_jsonl.exists() and out_jsonl.stat().st_size > 0:
        n = sum(1 for _ in open(out_jsonl, encoding="utf-8"))
        return {"status": "skipped", "chunks": n}

    rcept_no = report_meta["rcept_no"]
    xml_text, err = download_document(rcept_no)
    if err:
        return {"status": "error", "stage": "download", "msg": err}

    rk = cfg.report_kind_label(rt)
    (co_dir / f"{rk}_raw.xml").write_text(xml_text, encoding="utf-8")
    cleaned = clean_xml(xml_text)
    (co_dir / f"{rk}_cleaned.xml").write_text(cleaned, encoding="utf-8")

    try:
        root, mode = parse_with_fallback(cleaned)
    except Exception as e:
        return {"status": "error", "stage": "parse", "msg": f"{type(e).__name__}: {e}"}

    meta = {
        "group": co["group"], "stock_code": sc, "corp_code": co["corp_code"],
        "corp_name": name, "report_nm": report_meta.get("report_nm",""),
        "report_kind": rk, "report_type": rt,
        "rcept_no": rcept_no, "rcept_dt": report_meta.get("rcept_dt",""),
        "fiscal_period": rk,
        "source_url": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        "parse_mode": mode,
    }
    try:
        chunks = build_chunks(root, meta)
    except Exception as e:
        return {"status": "error", "stage": "chunk", "msg": f"{type(e).__name__}: {e}"}

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    return {"status": "ok", "chunks": len(chunks), "mode": mode}


# 메인 루프
results, failed = [], []
SKIP_EXISTING = True   # ← False 면 모두 재처리

try:
    from tqdm.auto import tqdm
    iterator = tqdm(mapped, desc="Companies")
except ImportError:
    iterator = mapped

for co in iterator:
    try:
        reports = fetch_reports(co["corp_code"])
    except Exception as e:
        failed.append({**co, "stage": "list", "error": str(e)})
        time.sleep(cfg.RATE_LIMIT_SLEEP); continue
    
    if reports.get("_status") != "000":
        print(f"  ⚠ {co['stock_code']} {co['name']}: {reports.get('_error')}")
    
    for rt in cfg.REPORT_TYPES:
        rm = reports.get(rt)
        if not rm:
            continue
        r = process_one_report(co, rt, rm, skip_existing=SKIP_EXISTING)
        row = {**co, "report_type": rt, "rcept_no": rm["rcept_no"], **r}
        results.append(row)
        if r["status"] == "error":
            failed.append(row)
    time.sleep(cfg.RATE_LIMIT_SLEEP)

print(f"\n완료: 처리 {len(results)}건, 실패 {len(failed)}건")


---
## 5. 결과 요약

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
if not df.empty:
    print("── 상태별 ──")
    print(df["status"].value_counts())
    print("\n── 보고서 종류별 ──")
    print(df.groupby(["report_type", "status"]).size().unstack(fill_value=0))
    
    cfg.LOG_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = cfg.LOG_DIR / f"progress_{datetime.now():%Y%m%d_%H%M%S}.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"\nsaved → {out_csv}")

if failed:
    print(f"\n── 실패 {len(failed)}건 ──")
    for f in failed[:10]:
        print(f"  {f.get('stock_code')} {f.get('name')} {f.get('report_type','-')}: "
              f"{f.get('msg') or f.get('error')}")
    out_fail = cfg.LOG_DIR / "failed.json"
    out_fail.write_text(json.dumps(failed, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n→ {out_fail}")

df.head(10) if not df.empty else "결과 없음"


---
## ✅ 다음 단계

JSONL 산출물은 **`config.py` 의 `RAG_DIR`** 폴더에 저장됨.

이 JSONL 들이 다음 단계 (임베딩 / 벡터DB / WACC 계산 등) 의 입력이 됩니다.